In [2]:
import json
import os
import requests
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

EMBED_FOLDER = "main_embedded_jsons"

json_files = sorted(f for f in os.listdir(EMBED_FOLDER) if f.endswith(".json"))

all_dfs = []
for file in json_files:
    with open(os.path.join(EMBED_FOLDER, file), encoding="utf-8") as f:
        data = json.load(f)

    temp = pd.DataFrame(data["chunks"])
    temp["video_number"] = data["video_number"]
    temp["title"] = data["title"]
    all_dfs.append(temp)

df = pd.concat(all_dfs, ignore_index=True)

# embeddings ko ek numpy matrix bana lo (search fast hogi)
embeddings = np.array(df["embedding"].tolist())

print("Total videos:", len(json_files))
print("Total chunks:", len(df))
print("Embedding shape:", embeddings.shape)

Total videos: 51
Total chunks: 2750
Embedding shape: (2750, 1024)


In [3]:
pd.set_option("display.max_colwidth", None)

def search(query, top_k=5):
    response = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": "bge-m3", "input": [query]}
    )
    response.raise_for_status()
    q_emb = response.json()["embeddings"][0]

    scores = cosine_similarity([q_emb], embeddings)[0]

    result = df.copy()
    result["similarity"] = scores
    result = (
        result.sort_values("similarity", ascending=False)
              .head(top_k)
              .reset_index(drop=True)
    )
    result.insert(0, "rank", result.index + 1)
    return result[["rank", "video_number", "title", "chunk_number",
                   "start", "end", "similarity", "text"]]


query = input("Ask a question: ")
print("Query:", query)

display(
    search(query)
    .style
    .background_gradient(subset=["similarity"], cmap="Greens")
    .format({"similarity": "{:.4f}"})
    .set_properties(subset=["text"], **{"text-align": "left", "white-space": "normal"})
    .hide(axis="index")
)

Ask a question:  which year python are developeed


Query: which year python are developeed


rank,video_number,title,chunk_number,start,end,similarity,text
1,1,Introduction to Programming & Python ｜ Python Tutorial,22,335.000000,351.000000,0.6213,"And in the video 1, we started working on the 1989 Mace. Python was developed by Python. It was not a single day. It was made by 1989. It was created in 1991."
2,1,Introduction to Programming & Python ｜ Python Tutorial,3,34.000000,47.000000,0.6092,Python is a dynamically typed object-oriented programming language. It was made in 1991 in Guida 1. Guida 1 was made in Monty Python. It is a show in the name of Python. When it was working on this language and they thought it was Python.
3,1,Introduction to Programming & Python ｜ Python Tutorial,23,351.000000,363.000000,0.5859,It was made in 1992. But it is true that it started in 1989. It was made in the period of years. What are the features of Python? The first thing I will tell you.
4,43,Virtual Environment in Python ｜ Python Tutorial,78,942.000000,952.000000,0.5200,Python. Python. Python. Python. Python.
5,43,Virtual Environment in Python ｜ Python Tutorial,76,920.000000,930.000000,0.5200,Python. Python. Python. Python. Python.
